# 01 — Bootstrap Colab Environment

Notebook generic của `colab-venv-bootstrap`. Đọc config từ `<repo>/colab/config.env` và build venv trên `/content/venvs/<slug>`.

**Khi chạy notebook này**:
- Lần đầu trên Colab (chưa có venv) → bootstrap script cài từ đầu (~10 phút).
- Sau khi `requirements.txt` đổi → fingerprint mismatch → tự rebuild.
- Runtime mới mà requirements không đổi → restore snapshot từ Drive (~30-60s).
- Đổi major Python/torch → set `FORCE_REINSTALL=1` ở cell cuối.

**Output**:
- `/content/venvs/<PROJECT_SLUG>/` — venv (local disk Colab).
- `<DRIVE_ROOT>/env/fingerprint.json` — metadata.
- `<DRIVE_ROOT>/env/venv_snapshot.tar.gz` — snapshot tái sử dụng.
- `<DRIVE_ROOT>/env/requirements.colab.lock.txt` — pip freeze.

**Notebook này hoàn toàn generic** — mọi config đến từ `colab/config.env`. Không có gì hardcode theo project.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import re

# ============================================================
# EDIT 2 biến này (match với notebook 00) để locate repo
# ============================================================
PROJECT_SLUG = 'my_project'
BASE_ROOT    = '/content/drive/MyDrive'

# ============================================================
# Derived
# ============================================================
REPO_DIR    = f'{BASE_ROOT}/{PROJECT_SLUG}'
CONFIG_FILE = f'{REPO_DIR}/colab/config.env'

if not os.path.isdir(REPO_DIR):
    raise FileNotFoundError(
        f'Repo không tồn tại tại {REPO_DIR}. Chạy 00_clone_repo.ipynb trước.'
    )

if not os.path.isfile(CONFIG_FILE):
    raise FileNotFoundError(
        f'Không tìm thấy {CONFIG_FILE}.\n'
        f'Đảm bảo bạn đã chạy colab/bootstrap/scripts/init_project.ps1 trong project gốc '
        f'để copy config.env từ templates/.'
    )

# Parse config.env (lightweight — chỉ đọc KEY="value" lines, bỏ qua comments)
config = {}
pattern = re.compile(r'^\s*([A-Z_][A-Z0-9_]*)\s*=\s*(.+?)\s*$')
with open(CONFIG_FILE, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.split('#', 1)[0]  # strip comment
        m = pattern.match(line)
        if m:
            k, v = m.group(1), m.group(2).strip()
            # Strip surrounding quotes
            if (v.startswith('"') and v.endswith('"')) or (v.startswith("'") and v.endswith("'")):
                v = v[1:-1]
            config[k] = v

# Verify config khớp 2 biến notebook
cfg_slug = config.get('PROJECT_SLUG', '')
cfg_base = config.get('BASE_ROOT', '')
if cfg_slug and cfg_slug != PROJECT_SLUG:
    print(f'[WARN] config.env có PROJECT_SLUG={cfg_slug!r} khác notebook ({PROJECT_SLUG!r})')
if cfg_base and cfg_base != BASE_ROOT:
    print(f'[WARN] config.env có BASE_ROOT={cfg_base!r} khác notebook ({BASE_ROOT!r})')

DRIVE_ROOT = f'{BASE_ROOT}/{PROJECT_SLUG}_colab'
VENV_DIR   = f'/content/venvs/{PROJECT_SLUG}'

os.makedirs(DRIVE_ROOT, exist_ok=True)
print(f'Repo dir   : {REPO_DIR}')
print(f'Config     : {CONFIG_FILE}')
print(f'Drive root : {DRIVE_ROOT}')
print(f'Venv dir   : {VENV_DIR}')
print()
print('=== Config from config.env (excerpt) ===')
for k in ('PROJECT_SLUG', 'BASE_ROOT', 'PYTHON_BIN', 'TORCH_INDEX_URL',
          'REQUIREMENTS_FILE', 'REUSE_CHECK_MODULES', 'FORCE_REINSTALL'):
    if k in config:
        print(f'  {k:<25s} = {config[k]}')

In [ ]:
# Kiểm tra GPU — bootstrap dùng TORCH_INDEX_URL từ config.env (CPU/cu117/cu121)
!nvidia-smi 2>&1 | head -10 || echo "[INFO] Không có GPU runtime — torch sẽ cài CPU build nếu config cho phép"

In [ ]:
%%bash -s "$REPO_DIR"
set -euo pipefail
REPO_DIR="$1"

BOOTSTRAP_SCRIPT="$REPO_DIR/colab/bootstrap/scripts/bootstrap_env.sh"

if [ ! -f "$BOOTSTRAP_SCRIPT" ]; then
    echo "[ERROR] Không tìm thấy $BOOTSTRAP_SCRIPT"
    echo "        Submodule colab/bootstrap chưa được pull. Chạy:"
    echo "          !git -C $REPO_DIR submodule update --init --recursive"
    exit 1
fi

chmod +x "$BOOTSTRAP_SCRIPT"
bash "$BOOTSTRAP_SCRIPT"

In [ ]:
%%bash -s "$VENV_DIR" "$DRIVE_ROOT"
set -euo pipefail
VENV="$1"
DRIVE_ROOT="$2"

echo '=== Python version ==='
"$VENV/bin/python" -V
echo
echo '=== Pip version ==='
"$VENV/bin/python" -m pip --version
echo
echo '=== Fingerprint ==='
cat "$DRIVE_ROOT/env/fingerprint.json" 2>/dev/null || echo '(no fingerprint yet)'
echo
echo '=== Lock file (top 20 packages) ==='
head -20 "$DRIVE_ROOT/env/requirements.colab.lock.txt" 2>/dev/null || echo '(no lock file yet)'
echo
echo '=== Snapshot size ==='
ls -lh "$DRIVE_ROOT/env/venv_snapshot.tar.gz" 2>/dev/null || echo '(no snapshot yet)'

## Force reinstall (chỉ chạy khi cần)

**Khi dùng**:
- Đổi torch/lightning version sang bản khác.
- Snapshot bị hỏng (extract fail, import lỗi lạ).
- Debug lỗi cài đặt.

Cell dưới đây mặc định **comment hết để an toàn**. Khi cần force rebuild: uncomment (bỏ `# ` đầu mỗi dòng) rồi chạy. Sau khi xong, comment lại để tránh chạy nhầm.

In [ ]:
# UNCOMMENT để force rebuild venv từ đầu (xóa snapshot + venv cũ)
# %%bash -s "$REPO_DIR" "$VENV_DIR" "$DRIVE_ROOT"
# set -euo pipefail
# REPO_DIR="$1"
# VENV_DIR="$2"
# DRIVE_ROOT="$3"
#
# echo "[INFO] Removing venv + snapshot for clean rebuild"
# rm -rf "$VENV_DIR"
# rm -f "$DRIVE_ROOT/env/venv_snapshot.tar.gz"
# rm -f "$DRIVE_ROOT/env/venv_snapshot.meta.json"
# rm -f "$DRIVE_ROOT/env/fingerprint.json"
#
# export FORCE_REINSTALL=1
# bash "$REPO_DIR/colab/bootstrap/scripts/bootstrap_env.sh"